In [1]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer einops addict easydict
!pip uninstall -y tensorflow protobuf
!pip install -q tensorflow protobuf 

In [2]:
import torch

torch.cuda.is_available()

True

In [3]:
%%capture
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR", local_dir="deepseek_ocr")

## Base model

In [4]:
from unsloth import FastVisionModel
import torch
from transformers import AutoModel
import os

os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit",  # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
]  # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit=False,  # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.7: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 6.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
!cp -r /kaggle/input/vnese-hwdb/hwdb_word_preprocessed /kaggle/working/hwdb_word_preprocessed

In [6]:
from datasets import load_from_disk, DatasetDict

dataset = load_from_disk(
    "/kaggle/working/hwdb_word_preprocessed",
)

In [7]:
# Split dataset: 80% train, 10% val, 10% test
# Train/val để finetune, test để evaluate sau này
train_testval = dataset.train_test_split(test_size=0.2, seed=42)
val_test = train_testval["test"].train_test_split(test_size=0.5, seed=42)

# Dataset cho training
finetune_dataset = DatasetDict(
    {"train": train_testval["train"], "val": val_test["train"]}
)

# Test set riêng
test_dataset = val_test["test"]

print("Dataset for finetuning:")
print(f"  Train: {len(finetune_dataset['train'])} samples")
print(f"  Val: {len(finetune_dataset['val'])} samples")
print(f"\nTest set (save riêng): {len(test_dataset)} samples")

Dataset for finetuning:
  Train: 88390 samples
  Val: 11049 samples

Test set (save riêng): 11049 samples


In [8]:
from tqdm import tqdm
from jiwer import cer, wer
import torch


def calculate_cer(predictions, references):
    """
    Calculate Character Error Rate (CER) for a list of predictions.
    """
    return cer(references, predictions)


def calculate_wer(predictions, references):
    """
    Calculate Word Error Rate (WER) for a list of predictions.
    """
    return wer(references, predictions)


def run_inference(model, tokenizer, dataset):
    """
    Run inference on the dataset using the provided model and tokenizer.
    """
    predictions = []
    ground_truths = []

    prompt = "<image>\nFree OCR."
    tmp_img_path = "temp_image.png"

    # Iterate through the dataset
    for i in tqdm(range(len(dataset)), desc="Running Inference"):
        item = dataset[i]
        image = item["image"]
        ground_truth = item["text"]

        # Save image temporarily
        image.save(tmp_img_path)

        result = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=tmp_img_path,
            output_path="./",
            base_size=1024,
            image_size=640,
            crop_mode=True,
            save_results=False,
            test_compress=False,
            eval_mode=True,
        )
        predictions.append(result.strip())
        ground_truths.append(ground_truth)

    return predictions, ground_truths


def evaluate_model(model, tokenizer, dataset, model_name="Model"):
    print(f"Evaluating {model_name}...")
    predictions, ground_truths = run_inference(model, tokenizer, dataset)

    cer_score = calculate_cer(predictions, ground_truths)
    wer_score = calculate_wer(predictions, ground_truths)

    results = {"CER": cer_score, "WER": wer_score}

    print(f"Results for {model_name}:")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")

    return results, predictions

# Baseline

In [9]:
# Evaluate Baseline Model
results_baseline, predictions_baseline = evaluate_model(
    model, tokenizer, test_dataset, model_name="Baseline"
)

# Clean up to save memory
del model
del tokenizer
torch.cuda.empty_cache()

Evaluating Baseline...


Running Inference: 100%|██████████| 11049/11049 [6:41:59<00:00,  2.18s/it]


Results for Baseline:
  CER: 23.5995
  WER: 2.8167


# Fine-tuned model

In [10]:
from unsloth import FastVisionModel
from transformers import AutoModel

# Path to the finetuned model
# Update this path to where your finetuned model is saved
FINETUNED_MODEL_PATH = "/kaggle/input/02-finetune/deepseek_ocr_finetuned"

if os.path.exists(FINETUNED_MODEL_PATH):
    print(f"Loading finetuned model from {FINETUNED_MODEL_PATH}...")
    finetuned_model, finetuned_tokenizer = FastVisionModel.from_pretrained(
        FINETUNED_MODEL_PATH,
        load_in_4bit=False,
        auto_model=AutoModel,
        trust_remote_code=True,
        unsloth_force_compile=True,
        use_gradient_checkpointing="unsloth",
    )

    results_finetuned, predictions_finetuned = evaluate_model(
        finetuned_model, finetuned_tokenizer, test_dataset, model_name="Finetuned"
    )
else:
    print(
        f"Finetuned model not found at {FINETUNED_MODEL_PATH}. Please check the path."
    )

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Loading finetuned model from /kaggle/input/02-finetune/deepseek_ocr_finetuned...


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.7: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 6.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating Finetuned...


Running Inference: 100%|██████████| 11049/11049 [3:44:18<00:00,  1.22s/it]

Results for Finetuned:
  CER: 0.4760
  WER: 0.7336


In [11]:
import pandas as pd

# Compare results
comparison_data = []

if "results_baseline" in locals():
    comparison_data.append({"Model": "Baseline", **results_baseline})

if "results_finetuned" in locals():
    comparison_data.append({"Model": "Finetuned", **results_finetuned})

if comparison_data:
    df_results = pd.DataFrame(comparison_data)
    print("\nComparison of Results:")
    print(df_results)

    # Calculate improvement if both exist
    if len(comparison_data) == 2:
        cer_improvement = results_baseline["CER"] - results_finetuned["CER"]
        wer_improvement = results_baseline["WER"] - results_finetuned["WER"]
        print(f"\nImprovement (Baseline - Finetuned):")
        print(f"  CER Improvement: {cer_improvement:.4f}")
        print(f"  WER Improvement: {wer_improvement:.4f}")
else:
    print("No results to compare.")


Comparison of Results:
       Model        CER       WER
0   Baseline  23.599466  2.816725
1  Finetuned   0.475978  0.733551

Improvement (Baseline - Finetuned):
  CER Improvement: 23.1235
  WER Improvement: 2.0832


In [12]:
import os
from datasets import Dataset

# Output directory for the Arrow dataset
output_dataset_path = "evaluation_results_dataset"

# Check if we have predictions
has_baseline = "predictions_baseline" in locals()
has_finetuned = "predictions_finetuned" in locals()

if not has_baseline and not has_finetuned:
    print("No predictions found to save.")
else:
    print(f"Saving results to {output_dataset_path}...")
    
    # Start with the test dataset (which already has images and ground truth)
    # We create a new dataset to avoid modifying the original one in place if we run this cell multiple times
    result_dataset = test_dataset
    
    # Add predictions as new columns
    if has_baseline:
        # Ensure length matches
        if len(predictions_baseline) == len(result_dataset):
            # Check if column already exists to avoid error on re-run
            if "baseline_prediction" in result_dataset.column_names:
                result_dataset = result_dataset.remove_columns("baseline_prediction")
            result_dataset = result_dataset.add_column("baseline_prediction", predictions_baseline)
        else:
            print(f"Warning: Baseline predictions length ({len(predictions_baseline)}) does not match dataset length ({len(result_dataset)}). Skipping.")

    if has_finetuned:
        # Ensure length matches
        if len(predictions_finetuned) == len(result_dataset):
            # Check if column already exists to avoid error on re-run
            if "finetuned_prediction" in result_dataset.column_names:
                result_dataset = result_dataset.remove_columns("finetuned_prediction")
            result_dataset = result_dataset.add_column("finetuned_prediction", predictions_finetuned)
        else:
            print(f"Warning: Finetuned predictions length ({len(predictions_finetuned)}) does not match dataset length ({len(result_dataset)}). Skipping.")

    # Save to disk
    result_dataset.save_to_disk(output_dataset_path)
    print(f"Saved dataset with {len(result_dataset)} samples to {output_dataset_path}")
    print(result_dataset)

Saving results to evaluation_results_dataset...


Flattening the indices:   0%|          | 0/11049 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11049 [00:00<?, ? examples/s]

Saved dataset with 11049 samples to evaluation_results_dataset
Dataset({
    features: ['image', 'text', 'baseline_prediction', 'finetuned_prediction'],
    num_rows: 11049
})
